In [1]:
## Imports ##
import pandas as pd
import requests
import pprint
import json

### Processing CSVs from OpenContext

In [2]:
## Trench data ##
# To use to find each individual trench's outliers
trench_URLs = pd.read_csv("trench-data.csv")["URI"]
#print(trench_URLs) #for debugging

trench_UUIDs = [s[32:] for s in trench_URLs]
#print(trench_UUIDs) #for debugging

### API Stuff

In [3]:
## API Requests ##
# Base URL, every trench's UUID will be appended to the end of it for API calls
# Is this an evil and stupid way to do this? Yea probably
api_url = "https://opencontext.org/utilities/geospace-outliers-within?item_id="

outliers = []

# Check every trench for outliers using API call (thanks Eric this is very convenient)
for uid in trench_UUIDs:

    try:
        response = requests.get(f"{api_url}{uid}", headers={    # I love Stack Overflow <3 Header fixed an issue I had
            'User-Agent': 'PostmanRuntime/7.43.4',              # https://stackoverflow.com/questions/76655144/connection-aborted-remotedisconnectedremote-end-closed-connection-without
        })  
    except:
        print("Unsuccessful connect.") 
        continue

    try:
        data = response.json()  # Convert result to JSON (because it is one)
    except:
        print(f"Invalid (empty) response encountered. UID={uid}")  # Temporary solution to errors
        continue

    # Get the list of outliers
    outlier_properties = data["child_geo_outliers"]

    for o in outlier_properties:
        outlier_entry = {
            "PC Number" : "",
            "URI" : "",
            "Longitude" : "",
            "Latitude" : "",
            "Long or Lat" : ""
        }

        # Insert data
        outlier_entry["PC Number"] = o["item__label"]
        outlier_entry["URI"] = o["uri"]
        outlier_entry["Longitude"] = o["longitude"]
        outlier_entry["Latitude"] = o["latitude"]

        # Outlier in which direction(s)
        long = False
        lat = False

        if o["flag__longitude"]:
            long = True
        if o["flag__latitude"]:
            lat = True

        long_lat_flags = (long, lat)
        outlier_entry["Long or Lat"] = long_lat_flags

        outliers.append(outlier_entry)


# WARNING TAKES 23 MINUTES TO RUN LMFAOOO
# Mag wifi is probably NOTTT helping
pprint.pprint(outliers)

Invalid (empty) response encountered. UID=a96fbda0-9054-4140-9622-cb07ce613758
Invalid (empty) response encountered. UID=05cab086-0e36-478c-f72c-22a15abcfb19
Invalid (empty) response encountered. UID=39e93c14-2dfe-4ff4-bcec-7a4f30de064a
Invalid (empty) response encountered. UID=2cea9c34-d5e1-4467-2ce7-27d269ba845c
Invalid (empty) response encountered. UID=4500ca4a-4b5a-41e8-5337-1c77990375af
Invalid (empty) response encountered. UID=44d03eae-759a-4c90-0b20-cab6bc8e010f
Invalid (empty) response encountered. UID=278f96d8-b34a-4b46-90ba-1c67a7a1a81d
Invalid (empty) response encountered. UID=5879c9ed-5fcf-4062-4477-fa5b44250143
Invalid (empty) response encountered. UID=1d751acc-a48d-4256-0898-2556e5280c56
Invalid (empty) response encountered. UID=e85194e8-b3f6-4c41-453a-c7ed3650dfb4
Invalid (empty) response encountered. UID=8db613b1-9160-4a26-1f84-ef45c8403e0a
Invalid (empty) response encountered. UID=0d24da34-1ace-4ea7-1a08-864c3771d57a
Invalid (empty) response encountered. UID=e651cbef-b

### Results

In [ ]:
## Saving results ##
filename = "Outlying Objects"
with open(filename, 'w') as f:
    f.write(json.dumps(outliers))
print(len(outliers))